In [1]:
import gc
# import torch

gc.collect()
# torch.cuda.empty_cache()
# torch.cuda.ipc_collect()

import os
os.listdir(); os.chdir("/aiau010_scratch/azm0269/clover/")

from clover.utils.utils import notebook_line_magic
notebook_line_magic()

In [45]:
import json
from pathlib import Path
import pandas as pd
import numpy as np


baselines = ["md3po", "ddpo", "b2diffurl", "dpok"]
eval_path = Path("outputs/")
training_data_eval_file = "training_metrics.json"
evaluation_data_eval_file = "eval_metrics.json"

def get_training_metrics_df(file_path):
    if file_path.exists():
        with open(file_path, "r") as f:
            list_of_metrics = json.load(f)
    else:
        print(f"File {file_path} does not exist. Skipping.")
        return pd.DataFrame()  # Return an empty DataFrame if the file doesn't exist
    all_metric_entries = []
    for entry in list_of_metrics:
        all_metric_entries.append(entry['metrics'])
    df = pd.DataFrame(all_metric_entries)
    return df

def get_eval_metrics_df(file_path):
    if file_path.exists():
        with open(file_path, "r") as f:
            list_of_metrics = json.load(f)
    else:
        print(f"File {file_path} does not exist. Skipping.")
        return pd.DataFrame()  # Return an empty DataFrame if the file doesn't exist
    
    scores = ['bert_reward', 'clip_reward']
    eval_summary = list()
    for entry in list_of_metrics:
        for score in scores:
            eval_summary.append({
                "value": round(np.array(entry[score]).mean(), 4),
                "score": score,
                "epoch": entry['epoch']
            })
    eval_df = pd.DataFrame(eval_summary)
    return eval_df
    
def get_metrics():
    training_eval_df = pd.DataFrame()
    eval_df = pd.DataFrame()
    for baseline in baselines:
        baseline_path = eval_path / baseline
        training_data_eval_file_path = baseline_path / "training_evals" / training_data_eval_file
        eval_data_file_path = baseline_path / "evals" / evaluation_data_eval_file

        
        df = get_training_metrics_df(training_data_eval_file_path)
        df["method"] = baseline
        training_eval_df = pd.concat([training_eval_df, df], ignore_index=True)

        df = get_eval_metrics_df(eval_data_file_path)
        df["method"] = baseline
        eval_df = pd.concat([eval_df, df], ignore_index=True)
    return training_eval_df, eval_df

In [46]:
training_eval_df, eval_df = get_metrics()


In [50]:
eval_summary = eval_df.pivot_table(
    index=['method', 'score'],
    columns='epoch',
    values='value',
)